In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from birddog.tracker import (
    PageTracker,
    DynamoDBPageChangeLogTable,
    DynamoDBPageTrackerTable,
    SQLitePageChangeLogTable,
    SQLitePageTrackerTable,
    PageChangeLog,
    )

from birddog.wiki import (
    get_recent_changes,
    lookup_namespace_id,
    )

from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import (
    DatabaseUpdater,
    )
from birddog.store import KeyValueStore, DynamoDBKeyValueStore
from birddog.utility import json_size, now, HeartbeatManager
#from birddog.log import get_logger

2026-02-05 14:13:08,028 [INFO] Translation is enabled. Using GCP translator
2026-02-05 14:13:08,029 [INFO] Using Google Cloud translation API
2026-02-05 14:13:08,029 [INFO] GoogleCloudTranslator using REST API
2026-02-05 14:13:08,038 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-02-05 14:13:08,411 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com


In [3]:
#def copy_page_tracker_to_ddb(batch_size=100, limit=None):
#    ddb_table = DynamoDBPageTrackerTable()
#    page_tracker = PageTracker()
#    entries = list(page_tracker._page_dict.items())
#    if not limit:
#        limit = len(entries)
#    print(f"pushing {limit} entries to DDB, batch_size={batch_size}")
#    for i in range(0, limit, batch_size):
#        print(f"batch {i}")
#        batch = { title: update for title, update in entries[i:(i+batch_size)] }
#        ddb_table.put(batch)

In [4]:
#copy_page_tracker_to_ddb(batch_size=1000)

In [5]:
#tracker = PageTracker()

In [6]:
#changes = PageChangeLog()

In [7]:
wikisource_file_ns = "Файл"
commons_file_ns = "File"
commons_base = "https://commons.wikimedia.org"

In [8]:
#lookup_namespace_id(wikisource_file_ns)

In [9]:
#lookup_namespace_id(commons_file_ns)

In [10]:
#c=get_recent_changes(cutoff_date="2026,02,03,23:00", base=commons_base, namespace=6)

In [11]:
#len(c)

In [12]:
#runtime = Runtime()

In [13]:
#updater = DatabaseUpdater(runtime)

In [14]:
from datetime import datetime, timezone
def now(universal=False):
    return datetime.now(timezone.utc) if universal else datetime.now()

In [70]:
_WIKIMEDIA_COMMONS = "https://commons.wikimedia.org"
_WIKI_DOC_TRACKER_KV_TABLE = "doc_tracker"
_WIKI_DOC_TRACKER_HEARTBEAT_INTERVAL = 15 # seconds
_WIKI_SENTINEL = "WIKI_SENTINEL"
_DOC_TABLE_SENTINEL = "DOC_SENTINEL"

class WikiDocTracker(HeartbeatManager):
    def __init__(
        self, 
        cutoff_time=None,
        base_url=_WIKIMEDIA_COMMONS, 
        namespace="File",
        db=None,
    ):
        self._base_url = base_url
        self._namespace = namespace
        self._db = db if db else Database()

        self._namespace_id = lookup_namespace_id(self._namespace)
        self._kv = KeyValueStore(table_name=_WIKI_DOC_TRACKER_KV_TABLE)
        self._doc_kv_namespace = f"{self._base_url}:{self._namespace}"
        self._sentinel_kv_namespace = f"{self._base_url}:{self._namespace}:SENTINELS"
        self._cutoff_time = cutoff_time
        if self._cutoff_time and not isinstance(self._cutoff_time, str):
            self._cutoff_time = str(self._cutoff_time)

        super().__init__(interval=_WIKI_DOC_TRACKER_HEARTBEAT_INTERVAL)

    def _reset(self):
        self._kv.remove_all(self._doc_kv_namespace)
        self._kv.remove_all(self._sentinel_kv_namespace)

    def _normalize_title(self, title):
        return title.replace(" ", "_")
        
    def _store_relevant_titles(self, records):
        relevant_titles = {
            d["title"]: d.get("link", "")
            for d in records 
            if d.get("link", "").startswith(self._base_url)
        }
        if relevant_titles:
            print(f"WikiDocTracker: inserting {len(relevant_titles)} relevant titles into kv store...")
            for title, link in relevant_titles.items():
                self._kv.insert(self._doc_kv_namespace, self._normalize_title(title), link)
        
    def _refresh_doc_titles(self):
        try:
            doc_sentinel = self._kv.get(self._sentinel_kv_namespace, _DOC_TABLE_SENTINEL)
        except KeyError:
            doc_sentinel = None
            
        # sort by descending creation date
        sort_spec = ("CreatedAt", False)

        oldest_creation_date = None
        newest_creation_date = None
        cursor = None
        while True:
            batch, cursor = self._db.scan("Documents", cursor=cursor, sort=sort_spec)
            #print(f"read {len(batch)} records")
            if not newest_creation_date:
                # save newest creation date on first batch (later batches are all older)
                newest_creation_date = max([rec["CreatedAt"] for rec in batch])
            self._store_relevant_titles(batch)
            #print(batch[-1])
            if not cursor or not batch or doc_sentinel and batch[-1]["CreatedAt"] < doc_sentinel:
                break

        if newest_creation_date:
            self._kv.insert(self._sentinel_kv_namespace, _DOC_TABLE_SENTINEL, str(newest_creation_date))

    def _get_all_docs(self):
        return { item[0]: item[1] for item in self._kv.get_all(self._doc_kv_namespace) }
            
    def _get_wiki_sentinel(self):
        try:
            t = self._kv.get(self._sentinel_kv_namespace, _WIKI_SENTINEL)
            if self._cutoff_time:
                t = max(t, self._cutoff_time)
            return t
        except KeyError:
            if self._cutoff_time:
                return self._cutoff_time
            raise ValueError("undefined wiki sentinel")

    def _set_wiki_sentinel(self, timestamp):
        t = self._kv.insert(self._sentinel_kv_namespace, _WIKI_SENTINEL, str(timestamp))

    def _clear_wiki_sentinel(self):
        try:
            self._kv.remove(self._sentinel_kv_namespace, _WIKI_SENTINEL)
        except KeyError:
            # ignore
            pass

    def _get_wiki_changes(self, cutoff):
        return get_recent_changes(utc_cutoff=cutoff, base=self._base_url, namespace=self._namespace_id)
        
    def heartbeat(self):
        next_sentinel = now(universal=True)
        last_sentinel = self._get_wiki_sentinel()
        print(f"WikiDocTracker: heartbeat start:\n"
              f"    t={next_sentinel}\n"
              f"    last={last_sentinel}"
             )

        changes = self._get_wiki_changes(last_sentinel)
        print(f"found {len(changes)} changes")

        self._refresh_doc_titles()
        docs = self._get_all_docs()

        doc_updates = []
        for title in changes.keys():
            print(f"checking change: {title}")
            doc_link = docs.get(self._normalize_title(title))
            if doc_link:
                print(f"found document update: {title}, {doc_link}")
                doc_updates.append(doc_link)

        if doc_updates:
            # todo refresh doc record in db
            pass

        self._set_wiki_sentinel(next_sentinel)
        print(f"WikiDocTracker: heartbeat finish")
        pass

In [71]:
wdt = WikiDocTracker(cutoff_time=now(universal=True))

2026-02-05 15:46:49,134 [INFO] fetch_url: 8 requests in last 60s → 0.13 req/s


In [72]:
#wdt._reset()

In [74]:
wdt.heartbeat()

WikiDocTracker: heartbeat start:
    t=2026-02-05 22:47:10.225701+00:00
    last=2026-02-05 22:46:50.720133+00:00
2026-02-05 15:47:10,462 [INFO] fetch_url: 8 requests in last 60s → 0.13 req/s
found 118 changes
checking change: File:20252530356 GOES18-GLM-FL-EXTENT3-EP112025-2000x2000.jpg
checking change: File:Asaka Lead Town あさかリードタウン・ネイバーズサークル.jpg
checking change: File:Drue Chrisman (49153249607) (cropped).jpg
checking change: File:EDDI 05mn 19850413.png
checking change: File:Cohoes Rail Trail - New York 031508 338.jpg
checking change: File:Sugar Sand Park playground, May 2008 (1).jpg
checking change: File:Statue of Alexander Stephens Clay - panoramio.jpg
checking change: File:ZevenOS RC1.png
checking change: File:Vivek Ramaswamy (53067677870).jpg
checking change: File:Shiawase-no-Ki-no-Mori 20220708 04.jpg
checking change: File:Asam amino Histidin.png
checking change: File:Marulan Railway Station Platform 1.jpg
checking change: File:Bangladesh Minority Janata Party flag.jpg
checking 

In [ ]:
wdt._namespace_id

In [ ]:
wdt._get_sentinel()

In [62]:
wdt._get_changes(wdt._get_wiki_sentinel())

2026-02-05 15:35:16,930 [INFO] fetch_url: 5 requests in last 60s → 0.08 req/s


{'File:Approaching Flaugergues Crater (bird’s-eye view) ESA517118.jpg': {'timestamp': '2026,02,05,22:35',
  'user': 'OptimusPrimeBot'},
 'File:EDDI 04wk 19851013.png': {'timestamp': '2026,02,05,22:35',
  'user': 'PantheraLeo1359531'},
 'File:20252521526 GOES18-GLM-FL-EXTENT3-EP112025-2000x2000.jpg': {'timestamp': '2026,02,05,22:35',
  'user': 'PantheraLeo1359531'},
 'File:Mercurius verandert Battus in een steen Metamorfosen van Ovidius (serietitel), RP-P-2014-67-36.jpg': {'timestamp': '2026,02,05,22:35',
  'user': 'SchlurcherBot'},
 'File:DJI OSMO with device holder.jpg': {'timestamp': '2026,02,05,22:35',
  'user': 'SchlurcherBot'},
 'File:Mapillary (dpwnaquiad) 2026-01-21 12H13M17S308 (1414960783587353 at hfQWkvTrqKM1CELe8YxVi4 with samsung SM-S711U).jpg': {'timestamp': '2026,02,05,22:35',
  'user': 'Emijrpbot'},
 'File:Kunstbauten I - 060c.jpg': {'timestamp': '2026,02,05,22:35',
  'user': 'Emijrpbot'},
 'File:Mogilyovskiye-gubernskiye-vedomosti-1859-20231110 134259.jpg': {'timestamp'

In [60]:
list(wdt._get_all_doc_titles())[:10]

['File:ДАХмО 227-2д-553 Книга запису народжених, одружених, розлучених і померлих євреїв по м. Брацлав,... (1851).pdf',
 'File:ДАЧгО Р-9003-1-105 Книга державної реєстрації поновлених актів про народження (1947).pdf',
 'File:ДАВоО Р-3247-2-491 Книга реєстрації актів про народження Любомльський район (1944).pdf',
 'File:ДАВоО Р-3247-2-1885 Книга реєстрації актів про народження Старовижівський район (1949).pdf',
 'File:ДАЧгО Р-8998-1-87 Книга реєстрації актів про смерть (1934).pdf',
 'File:ДАВоО Р-3247-2-1932 Книга реєстрації актів про народження (поновлені) Луківський район (1949).pdf',
 'File:ДАВоО Р-3247-2-905 Книга реєстрації актів про смерть Луцьк (1945).pdf',
 'File:ДАВоО Р-3247-2-1660 Книга державної реєстрації народжень Берестечківський район (1949).pdf',
 'File:ДАВоО Р-3247-2-1473 Книга реєстрації актів про народження Ковельський район (1948).pdf',
 'File:ДАХмО 227-2д-633 Виписка з книги для запису померлих євреїв по м. Піщанка (1852).pdf']

In [ ]:
wdt.start()

In [ ]:
wdt.stop()

In [ ]:
wdt._clear_sentinel()

In [ ]:
wdt._get_sentinel()

In [ ]:
now(universal=True)

In [ ]:
def _normalize_date_string(s):
    return str(datetime.fromisoformat(s.replace("Z", "+00:00")))


In [ ]:
_normalize_date_string(str(now()))

In [ ]:
wdt._db.scan("Documents", where=("timestamp", "ge", _normalize_date_string(str(now(universal=True)))))

In [ ]:
wdt._bootstrap_doc_titles()

In [ ]:
t=wdt._kv.get_all(wdt._kv_namespace)

In [26]:
#wdt._db.scan("Documents", sort=("CreatedAt", False))

In [23]:
wdt._db.write("Documents", {"title": "foo", "link": "bar"})

2026-02-05 14:20:00,532 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s


18599

In [24]:
wdt._db.read("Documents", 18599)

2026-02-05 14:20:26,254 [INFO] fetch_url: 3 requests in last 60s → 0.05 req/s


{'Id': 18599,
 'CreatedAt': '2026-02-05 21:20:00+00:00',
 'UpdatedAt': None,
 'title': 'foo',
 'link': 'bar',
 'processor': None,
 'pages_processed': None,
 'doc_type': [],
 'content_code': None,
 'process_code': None,
 'comments': None,
 'birddog_alert': False,
 'source': None,
 'timestamp': None,
 'byte_size': None,
 'mimetype': None,
 'mediatype': None,
 'width': None,
 'height': None,
 'page_count': None,
 'sha1_hash': None,
 'description_url': None,
 'thumb_width': None,
 'thumb_height': None,
 'thumb_url': None,
 'owning_pages': 0,
 'availability': None,
 'import_message': None,
 '_nc_m2m_Pages_Documents': [],
 'root_label': [],
 'root': [],
 'label': []}

In [ ]:
db = Database()

In [ ]:
dids = db.get_all_ids("Documents")

In [ ]:
doc_recs = db.read("Documents", dids)

In [ ]:
commons_prefix = "https://commons.wikimedia.org/wiki/File:"

In [ ]:
[d["title"] for d in doc_recs[:10] if d.get("link", "").startswith(commons_prefix)]

In [ ]:
commons_titles = { d["title"]: {"Id": d.get("Id"), "timestamp": d.get("timestamp")} 
                   for d in doc_recs 
                   if d.get("link", "").startswith(commons_prefix)
                 }

In [ ]:
#commons_titles

In [ ]:
commons_changes = [
    change for change in c 
    if change[0] in commons_titles
    ]

In [ ]:
commons_changes

In [ ]:
doc_store = KeyValueStore(table_name="doc_titles")
doc_ns = "commons"

In [ ]:
list(commons_titles.items())[:1]

In [ ]:
len(commons_titles)

In [ ]:
def store_titles(store, titles, ns=doc_ns):
    for title, entry in titles.items():
        store.insert(ns, title, str(entry.get("Id", "")))

In [ ]:
store_titles(doc_store, commons_titles)

In [ ]:
t = doc_store.get_all(doc_ns)